# Advanced Problems with Solutions: Python Attribute Write Accessors

This notebook is a **problem-driven expansion** of the supplied lesson on attribute writes.

## Scope

The source lesson centers on:

- intercepting instance writes with `__setattr__`;
- intercepting class writes with a metaclass;
- the interaction between `__setattr__` and **data descriptors**;
- the difference between data and non-data descriptors during assignment;
- avoiding infinite recursion by delegating to `super().__setattr__` (or `object.__setattr__`).

> Terminology note: the supplied prose mentions `__setattribute__`, while its working code correctly uses Python's `__setattr__` hook. This notebook uses `__setattr__`.

## How to use this notebook

For each problem:

1. Read the challenge.
2. Predict the behavior before running code.
3. Implement your own version if you want practice.
4. Compare with the supplied solution.
5. Run the verification assertions.

All examples use the Python standard library only.

## Best-practice reference

When overriding `__setattr__`:

- Prefer `super().__setattr__(name, value)` when cooperative inheritance matters.
- Prefer `object.__setattr__(self, name, value)` when you intentionally need the base implementation directly.
- Never write `self.some_attr = ...` inside `__setattr__` unless you deliberately want to re-enter the hook.
- Use `object.__getattribute__` carefully when reading internal state from inside hooks.
- Keep policy checks small and explicit.
- Remember that **data descriptors** (objects defining `__set__` or `__delete__`) participate in assignment.
- Treat metaclass `__setattr__` as class-object policy, not instance policy.
- Use tests for allowed writes, rejected writes, initialization, inheritance, descriptors, and internal bookkeeping.

# Problem 1 — Trace the complete write path

Create a class that logs instance writes and a metaclass that logs class writes.

Requirements:

- `obj.x = 10` must be logged by the instance hook.
- `ClassName.x = 10` must be logged by the metaclass hook.
- Initialization must still work.
- Store events in separate logs so the order can be asserted.

In [1]:
class P01Meta(type):
    class_events = []

    def __setattr__(cls, name, value):
        if name != "class_events":
            P01Meta.class_events.append((cls.__name__, name, value))
        return super().__setattr__(name, value)


class P01Record(metaclass=P01Meta):
    def __init__(self, name):
        object.__setattr__(self, "instance_events", [])
        self.name = name

    def __setattr__(self, name, value):
        if name != "instance_events":
            self.instance_events.append((name, value))
        super().__setattr__(name, value)


P01Meta.class_events.clear()
r = P01Record("Ada")
r.score = 99
P01Record.category = "student"

assert r.instance_events == [("name", "Ada"), ("score", 99)]
assert P01Meta.class_events == [("P01Record", "category", "student")]
print("instance:", r.instance_events)
print("class:", P01Meta.class_events)

instance: [('name', 'Ada'), ('score', 99)]
class: [('P01Record', 'category', 'student')]


**Why it works**

`r.score = 99` dispatches through `P01Record.__setattr__`.  
`P01Record.category = ...` is an assignment to the **class object**, so its type (`P01Meta`) controls that write.

# Problem 2 — Repair an infinite-recursion bug

The following design is wrong in principle:

```python
def __setattr__(self, name, value):
    self.__dict__[name] = value
```

It may work in some simple cases, but direct `__dict__` manipulation bypasses descriptor behavior and can fail with slotted objects.

Build a robust setter that:

- rejects names starting with exactly one underscore;
- allows normal public attributes;
- allows double-underscore-style names;
- delegates using the normal object protocol.

In [2]:
class P02Protected:
    def __setattr__(self, name, value):
        single_private = name.startswith("_") and not name.startswith("__")
        if single_private:
            raise AttributeError(f"{name!r} is read-only")
        super().__setattr__(name, value)


p = P02Protected()
p.answer = 42
p.__special__ = "allowed"

try:
    p._secret = "blocked"
except AttributeError as exc:
    print(type(exc).__name__, exc)

assert p.answer == 42
assert p.__special__ == "allowed"
assert "_secret" not in p.__dict__

AttributeError '_secret' is read-only


# Problem 3 — Normalize data on assignment

Implement a `User` model where:

- `username` is stripped and lowercased;
- `email` is stripped and lowercased;
- `age` must be a non-boolean integer in `[0, 150]`;
- unknown attributes are still allowed.

Do not duplicate storage logic.

In [3]:
class P03User:
    def __setattr__(self, name, value):
        if name == "username":
            if not isinstance(value, str):
                raise TypeError("username must be str")
            value = value.strip().lower()
            if not value:
                raise ValueError("username cannot be empty")

        elif name == "email":
            if not isinstance(value, str):
                raise TypeError("email must be str")
            value = value.strip().lower()
            if "@" not in value:
                raise ValueError("email must contain @")

        elif name == "age":
            if isinstance(value, bool) or not isinstance(value, int):
                raise TypeError("age must be an integer")
            if not 0 <= value <= 150:
                raise ValueError("age out of range")

        super().__setattr__(name, value)


u = P03User()
u.username = "  Ada_Lovelace "
u.email = " ADA@Example.COM "
u.age = 36
u.note = "free-form"

assert u.username == "ada_lovelace"
assert u.email == "ada@example.com"
assert u.age == 36
assert u.note == "free-form"
print(u.__dict__)

{'username': 'ada_lovelace', 'email': 'ada@example.com', 'age': 36, 'note': 'free-form'}


# Problem 4 — Data descriptor vs non-data descriptor

Construct:

- a non-data descriptor defining only `__get__`;
- a data descriptor defining `__get__` and `__set__`;
- a host class that also overrides `__setattr__`.

Show that assignment to the data descriptor invokes its `__set__`, while assignment to the non-data descriptor becomes a normal instance attribute.

In [4]:
class P04NonData:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return "value-from-non-data-descriptor"


class P04Data:
    def __set_name__(self, owner, name):
        self.storage_name = f"_{name}_storage"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.storage_name)

    def __set__(self, instance, value):
        instance.__dict__[self.storage_name] = value * 10


class P04Host:
    non_data = P04NonData()
    data = P04Data()

    def __init__(self):
        object.__setattr__(self, "writes", [])

    def __setattr__(self, name, value):
        if name != "writes":
            self.writes.append((name, value))
        super().__setattr__(name, value)


h = P04Host()
h.data = 7
h.non_data = 8

assert h.data == 70
assert h.non_data == 8
assert h.__dict__["_data_storage"] == 70
assert h.__dict__["non_data"] == 8
assert h.writes == [("data", 7), ("non_data", 8)]
print(h.__dict__)

{'writes': [('data', 7), ('non_data', 8)], '_data_storage': 70, 'non_data': 8}


# Problem 5 — Build a reusable validating descriptor

Create a descriptor that:

- knows its public name through `__set_name__`;
- stores under a private key;
- validates type and optional bounds;
- gives a useful error if read before assignment.

Use it for a `Product(price, quantity)` class.

In [5]:
class P05Validated:
    def __init__(self, expected_type, *, min_value=None, max_value=None):
        self.expected_type = expected_type
        self.min_value = min_value
        self.max_value = max_value

    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{owner.__name__}__{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.storage_name]
        except KeyError:
            raise AttributeError(f"{self.name!r} has not been set") from None

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, self.expected_type):
            raise TypeError(
                f"{self.name} must be {self.expected_type.__name__}"
            )
        if self.min_value is not None and value < self.min_value:
            raise ValueError(f"{self.name} must be >= {self.min_value}")
        if self.max_value is not None and value > self.max_value:
            raise ValueError(f"{self.name} must be <= {self.max_value}")
        instance.__dict__[self.storage_name] = value


class P05Product:
    price = P05Validated((int, float), min_value=0)
    quantity = P05Validated(int, min_value=0)

    def __init__(self, price, quantity):
        self.price = price
        self.quantity = quantity


product = P05Product(12.5, 4)
assert product.price == 12.5
assert product.quantity == 4

try:
    product.quantity = -1
except ValueError as exc:
    print(type(exc).__name__, exc)

print(product.__dict__)

ValueError quantity must be >= 0
{'_P05Product__price': 12.5, '_P05Product__quantity': 4}


# Problem 6 — Property setters are data descriptors too

Demonstrate that a `property` with a setter participates in the same assignment machinery as a custom data descriptor.

Add an outer `__setattr__` logger and prove that:

1. the outer hook sees the raw assignment;
2. `super().__setattr__` delegates to the property setter;
3. the setter stores a normalized backing value.

In [6]:
class P06Account:
    def __init__(self, owner):
        object.__setattr__(self, "events", [])
        self.owner = owner

    @property
    def owner(self):
        return self._owner

    @owner.setter
    def owner(self, value):
        if not isinstance(value, str):
            raise TypeError("owner must be str")
        object.__setattr__(self, "_owner", value.strip().title())

    def __setattr__(self, name, value):
        if name != "events":
            self.events.append((name, value))
        super().__setattr__(name, value)


a = P06Account("  ada lovelace ")
assert a.owner == "Ada Lovelace"
assert a.events == [("owner", "  ada lovelace ")]
print(a.events, a.__dict__)

[('owner', '  ada lovelace ')] {'events': [('owner', '  ada lovelace ')], '_owner': 'Ada Lovelace'}


# Problem 7 — Freeze an object after initialization

Implement a class that is mutable during `__init__`, then rejects:

- adding a new attribute after initialization;
- changing any existing public attribute after initialization.

Private bookkeeping may still be modified internally.

In [7]:
class P07FrozenAfterInit:
    def __init__(self, **fields):
        object.__setattr__(self, "_frozen", False)
        for name, value in fields.items():
            setattr(self, name, value)
        object.__setattr__(self, "_frozen", True)

    def __setattr__(self, name, value):
        frozen = object.__getattribute__(self, "_frozen")
        if frozen and not name.startswith("_"):
            raise AttributeError("object is frozen")
        super().__setattr__(name, value)

    def _set_internal(self, name, value):
        if not name.startswith("_"):
            raise ValueError("internal names must begin with _")
        object.__setattr__(self, name, value)


cfg = P07FrozenAfterInit(host="localhost", port=8080)
cfg._set_internal("_cache_ready", True)

for name, value in [("host", "example.com"), ("debug", True)]:
    try:
        setattr(cfg, name, value)
    except AttributeError as exc:
        print(f"{name}: {exc}")

assert cfg.host == "localhost"
assert cfg._cache_ready is True

host: object is frozen
debug: object is frozen


# Problem 8 — Allow existing fields but reject new fields

This is subtler than full freezing.

After initialization:

- existing public fields may change;
- unknown public fields are rejected;
- internal attributes beginning with `_` are allowed.

Do not hard-code the field names.

In [8]:
class P08SealedShape:
    def __init__(self, **fields):
        object.__setattr__(self, "_sealed", False)
        for name, value in fields.items():
            setattr(self, name, value)
        object.__setattr__(self, "_sealed", True)

    def __setattr__(self, name, value):
        sealed = object.__getattribute__(self, "_sealed")
        if sealed and not name.startswith("_") and name not in self.__dict__:
            raise AttributeError(f"cannot add new field {name!r}")
        super().__setattr__(name, value)


point = P08SealedShape(x=1, y=2)
point.x = 10
point._memo = "ok"

try:
    point.z = 3
except AttributeError as exc:
    print(type(exc).__name__, exc)

assert point.x == 10
assert point.y == 2
assert point._memo == "ok"
assert "z" not in point.__dict__

AttributeError cannot add new field 'z'


# Problem 9 — Audit every successful public write

Requirements:

- record `(name, old_value, new_value)` after each successful public assignment;
- do not record failed validation;
- do not record writes to the audit log itself;
- use a sentinel so "attribute did not exist" is distinguishable from a stored `None`.

In [9]:
_MISSING = object()

class P09Audited:
    def __init__(self):
        object.__setattr__(self, "_audit", [])

    @property
    def audit(self):
        return tuple(self._audit)

    def validate(self, name, value):
        if name == "balance" and value < 0:
            raise ValueError("balance cannot be negative")

    def __setattr__(self, name, value):
        if name.startswith("_"):
            return super().__setattr__(name, value)

        old = self.__dict__.get(name, _MISSING)
        self.validate(name, value)  # may raise; no event is recorded
        super().__setattr__(name, value)
        self._audit.append((name, old, value))


acct = P09Audited()
acct.balance = 100
acct.balance = 75
acct.note = None

try:
    acct.balance = -1
except ValueError:
    pass

assert acct.balance == 75
assert len(acct.audit) == 3
assert acct.audit[0][0] == "balance"
assert acct.audit[0][1] is _MISSING
assert acct.audit[1] == ("balance", 100, 75)
print(acct.audit)

(('balance', <object object at 0x0000026E9CB39430>, 100), ('balance', 100, 75), ('note', <object object at 0x0000026E9CB39430>, None))


# Problem 10 — Undo the most recent assignment

Extend the audit idea into an undoable object.

Requirements:

- store enough history to restore a previous value;
- if an attribute did not previously exist, undo must delete it;
- undo itself must not create a new history entry;
- raise `IndexError` if there is nothing to undo.

In [10]:
_P10_MISSING = object()

class P10Undoable:
    def __init__(self):
        object.__setattr__(self, "_history", [])
        object.__setattr__(self, "_recording", True)

    def __setattr__(self, name, value):
        if name.startswith("_"):
            return object.__setattr__(self, name, value)

        old = self.__dict__.get(name, _P10_MISSING)
        object.__setattr__(self, name, value)

        if object.__getattribute__(self, "_recording"):
            self._history.append((name, old))

    def undo(self):
        if not self._history:
            raise IndexError("nothing to undo")

        name, old = self._history.pop()
        object.__setattr__(self, "_recording", False)
        try:
            if old is _P10_MISSING:
                object.__delattr__(self, name)
            else:
                object.__setattr__(self, name, old)
        finally:
            object.__setattr__(self, "_recording", True)


obj = P10Undoable()
obj.x = 1
obj.x = 2
obj.y = 3

obj.undo()
assert not hasattr(obj, "y")

obj.undo()
assert obj.x == 1

obj.undo()
assert not hasattr(obj, "x")

print(obj.__dict__)

{'_history': [], '_recording': True}


# Problem 11 — Enforce a schema with per-field validators

Build a base model where subclasses declare a `schema` dictionary mapping field names to validator functions.

Rules:

- public attributes not in the schema are rejected;
- validators return the normalized value to store;
- private attributes remain available for implementation details.

In [11]:
def p11_nonempty_text(value):
    if not isinstance(value, str):
        raise TypeError("expected text")
    value = value.strip()
    if not value:
        raise ValueError("text cannot be empty")
    return value

def p11_positive_int(value):
    if isinstance(value, bool) or not isinstance(value, int):
        raise TypeError("expected integer")
    if value <= 0:
        raise ValueError("must be positive")
    return value


class P11SchemaModel:
    schema = {}

    def __setattr__(self, name, value):
        if name.startswith("_"):
            return super().__setattr__(name, value)

        try:
            validator = type(self).schema[name]
        except KeyError:
            raise AttributeError(f"unknown field {name!r}") from None

        normalized = validator(value)
        super().__setattr__(name, normalized)


class P11Book(P11SchemaModel):
    schema = {
        "title": p11_nonempty_text,
        "pages": p11_positive_int,
    }

    def __init__(self, title, pages):
        self.title = title
        self.pages = pages


book = P11Book("  Python Internals  ", 500)
assert book.title == "Python Internals"
assert book.pages == 500

try:
    book.author = "Unknown"
except AttributeError as exc:
    print(type(exc).__name__, exc)

AttributeError unknown field 'author'


# Problem 12 — Cooperative inheritance with multiple `__setattr__` mixins

Implement two mixins:

- one trims all assigned strings;
- one rejects `None` for public attributes.

Combine them with a model class.

The key requirement is **cooperative `super()`**: every mixin should perform one responsibility and forward the write exactly once.

In [12]:
class P12TrimStringsMixin:
    def __setattr__(self, name, value):
        if isinstance(value, str):
            value = value.strip()
        super().__setattr__(name, value)


class P12NoNoneMixin:
    def __setattr__(self, name, value):
        if not name.startswith("_") and value is None:
            raise ValueError(f"{name} cannot be None")
        super().__setattr__(name, value)


class P12Customer(P12TrimStringsMixin, P12NoNoneMixin):
    def __init__(self, name, city):
        self.name = name
        self.city = city


customer = P12Customer("  Ada  ", "  London ")
assert customer.name == "Ada"
assert customer.city == "London"

try:
    customer.city = None
except ValueError as exc:
    print(type(exc).__name__, exc)

print(P12Customer.__mro__)

ValueError city cannot be None
(<class '__main__.P12Customer'>, <class '__main__.P12TrimStringsMixin'>, <class '__main__.P12NoNoneMixin'>, <class 'object'>)


# Problem 13 — Validate class attributes in a metaclass

Create a metaclass policy for configuration classes:

- any class attribute whose name starts with `MAX_` must be a non-negative integer;
- regular class attributes are unrestricted;
- instance writes are unaffected unless the class separately overrides `__setattr__`.

In [13]:
class P13ConfigMeta(type):
    def __setattr__(cls, name, value):
        if name.startswith("MAX_"):
            if isinstance(value, bool) or not isinstance(value, int):
                raise TypeError(f"{name} must be int")
            if value < 0:
                raise ValueError(f"{name} must be non-negative")
        super().__setattr__(name, value)


class P13Settings(metaclass=P13ConfigMeta):
    MAX_RETRIES = 3


P13Settings.MAX_RETRIES = 5
P13Settings.description = "runtime configuration"

try:
    P13Settings.MAX_RETRIES = -1
except ValueError as exc:
    print(type(exc).__name__, exc)

instance = P13Settings()
instance.MAX_RETRIES = -100  # instance attribute: metaclass hook is not involved

assert P13Settings.MAX_RETRIES == 5
assert instance.MAX_RETRIES == -100

ValueError MAX_RETRIES must be non-negative


# Problem 14 — Prevent mutation of selected class constants

Use a metaclass to make names in `__class_constants__` write-once after class creation.

The class body must be able to define them initially, but later assignments such as `Physics.C = ...` must fail.

In [14]:
class P14ConstantsMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        locked = frozenset(namespace.get("__class_constants__", ()))
        type.__setattr__(cls, "__locked_class_constants__", locked)
        return cls

    def __setattr__(cls, name, value):
        locked = cls.__dict__.get("__locked_class_constants__", frozenset())
        if name in locked and name in cls.__dict__:
            raise AttributeError(f"class constant {name!r} is immutable")
        super().__setattr__(name, value)


class P14Physics(metaclass=P14ConstantsMeta):
    __class_constants__ = {"C", "G"}
    C = 299_792_458
    G = 6.67430e-11


P14Physics.unit = "SI"

try:
    P14Physics.C = 1
except AttributeError as exc:
    print(type(exc).__name__, exc)

assert P14Physics.C == 299_792_458
assert P14Physics.unit == "SI"

AttributeError class constant 'C' is immutable


# Problem 15 — `__slots__` plus `__setattr__`

Build a slotted point with `x` and `y` only.

Requirements:

- no instance `__dict__`;
- assignments are validated as real numbers;
- unknown attributes fail naturally;
- no direct `__dict__` storage tricks.

In [15]:
from numbers import Real

class P15Point:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __setattr__(self, name, value):
        if name in {"x", "y"}:
            if isinstance(value, bool) or not isinstance(value, Real):
                raise TypeError(f"{name} must be a real number")
        super().__setattr__(name, value)


pt = P15Point(1, 2.5)
pt.x = -3

try:
    pt.z = 10
except AttributeError as exc:
    print(type(exc).__name__, exc)

assert pt.x == -3
assert pt.y == 2.5
assert not hasattr(pt, "__dict__")

AttributeError 'P15Point' object has no attribute 'z'


# Problem 16 — Dataclass interaction

A dataclass-generated `__init__` performs normal assignments, so an overridden `__setattr__` sees those writes too.

Create a dataclass whose `name` field is normalized and whose `score` is restricted to `[0, 100]`.

In [16]:
from dataclasses import dataclass

@dataclass
class P16Student:
    name: str
    score: int

    def __setattr__(self, name, value):
        if name == "name":
            if not isinstance(value, str):
                raise TypeError("name must be str")
            value = value.strip().title()

        if name == "score":
            if isinstance(value, bool) or not isinstance(value, int):
                raise TypeError("score must be int")
            if not 0 <= value <= 100:
                raise ValueError("score must be in [0, 100]")

        super().__setattr__(name, value)


student = P16Student("  ada lovelace ", 98)
assert student.name == "Ada Lovelace"
assert student.score == 98

student.score = 100
assert student.score == 100

try:
    student.score = 101
except ValueError as exc:
    print(type(exc).__name__, exc)

ValueError score must be in [0, 100]


# Problem 17 — Avoid double validation between `__setattr__` and a descriptor

A common design mistake validates a field in both `__setattr__` and its descriptor.

Instead:

- let `__setattr__` handle cross-cutting concerns such as logging;
- let the descriptor own field-specific validation.

Prove each assignment is validated exactly once.

In [17]:
class P17Positive:
    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"
        self.validation_count = 0

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.storage_name]

    def __set__(self, instance, value):
        self.validation_count += 1
        if value <= 0:
            raise ValueError(f"{self.name} must be positive")
        instance.__dict__[self.storage_name] = value


class P17Order:
    quantity = P17Positive()

    def __init__(self, quantity):
        object.__setattr__(self, "_writes", [])
        self.quantity = quantity

    def __setattr__(self, name, value):
        if not name.startswith("_"):
            self._writes.append((name, value))
        super().__setattr__(name, value)


P17Order.quantity.validation_count = 0
order = P17Order(2)
order.quantity = 5

assert order.quantity == 5
assert order._writes == [("quantity", 2), ("quantity", 5)]
assert P17Order.quantity.validation_count == 2
print("validations:", P17Order.quantity.validation_count)

validations: 2


# Problem 18 — Descriptor storage without polluting `__dict__`

Use `weakref.WeakKeyDictionary` so a descriptor can store per-instance values externally.

This is useful when:

- you do not want descriptor values in `instance.__dict__`;
- instances support weak references;
- descriptor lifetime should not keep instances alive.

In [18]:
from weakref import WeakKeyDictionary
import gc

class P18ExternalStorage:
    def __init__(self):
        self._values = WeakKeyDictionary()

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if instance not in self._values:
            raise AttributeError(self.name)
        return self._values[instance]

    def __set__(self, instance, value):
        self._values[instance] = value


class P18Session:
    token = P18ExternalStorage()


session = P18Session()
session.token = "abc123"

assert session.token == "abc123"
assert "token" not in session.__dict__
assert len(P18Session.token._values) == 1

del session
gc.collect()

assert len(P18Session.token._values) == 0
print("external entries after GC:", len(P18Session.token._values))

external entries after GC: 0


# Problem 19 — Transactional multi-field update with rollback

Create an object that supports:

```python
obj.update_atomic(a=..., b=..., c=...)
```

If any assignment fails:

- every earlier assignment in that same update is rolled back;
- attributes that did not previously exist are removed;
- rollback must bypass normal validation/history logic.

In [19]:
_P19_MISSING = object()

class P19Profile:
    def __init__(self, *, age=0, name="unknown"):
        self.age = age
        self.name = name

    def __setattr__(self, name, value):
        if name == "age":
            if isinstance(value, bool) or not isinstance(value, int):
                raise TypeError("age must be int")
            if value < 0:
                raise ValueError("age must be non-negative")
        if name == "name":
            if not isinstance(value, str) or not value.strip():
                raise ValueError("name must be non-empty text")
            value = value.strip()
        super().__setattr__(name, value)

    def update_atomic(self, **changes):
        before = {
            name: self.__dict__.get(name, _P19_MISSING)
            for name in changes
        }

        try:
            for name, value in changes.items():
                setattr(self, name, value)
        except Exception:
            for name, old in before.items():
                if old is _P19_MISSING:
                    if name in self.__dict__:
                        object.__delattr__(self, name)
                else:
                    object.__setattr__(self, name, old)
            raise


profile = P19Profile(age=30, name="Ada")

try:
    profile.update_atomic(name="Grace", city="London", age=-1)
except ValueError as exc:
    print(type(exc).__name__, exc)

assert profile.name == "Ada"
assert profile.age == 30
assert not hasattr(profile, "city")

profile.update_atomic(name="Grace", city="Arlington", age=40)
assert profile.name == "Grace"
assert profile.city == "Arlington"
assert profile.age == 40
print(profile.__dict__)

ValueError age must be non-negative
{'age': 40, 'name': 'Grace', 'city': 'Arlington'}


# Problem 20 — Internal cache writes that must bypass policy

Build a model where public attributes are immutable after creation, but a computed property is allowed to populate a private cache.

This illustrates a justified use of `object.__setattr__` for internal state.

In [20]:
class P20Vector:
    def __init__(self, x, y):
        object.__setattr__(self, "_initializing", True)
        self.x = x
        self.y = y
        object.__setattr__(self, "_initializing", False)
        object.__setattr__(self, "_magnitude_cache", None)

    def __setattr__(self, name, value):
        if not name.startswith("_"):
            initializing = object.__getattribute__(self, "_initializing")
            if not initializing:
                raise AttributeError("public coordinates are immutable")
        super().__setattr__(name, value)

    @property
    def magnitude(self):
        cached = object.__getattribute__(self, "_magnitude_cache")
        if cached is None:
            value = (self.x ** 2 + self.y ** 2) ** 0.5
            object.__setattr__(self, "_magnitude_cache", value)
            return value
        return cached


v = P20Vector(3, 4)
assert v.magnitude == 5.0
assert v._magnitude_cache == 5.0

try:
    v.x = 100
except AttributeError as exc:
    print(type(exc).__name__, exc)

AttributeError public coordinates are immutable


# Problem 21 — Make a write-once descriptor

Implement a descriptor whose managed field may be assigned only once per instance.

This is different from freezing the entire object: other fields can remain mutable.

In [21]:
class P21WriteOnce:
    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.storage_name not in instance.__dict__:
            raise AttributeError(self.name)
        return instance.__dict__[self.storage_name]

    def __set__(self, instance, value):
        if self.storage_name in instance.__dict__:
            raise AttributeError(f"{self.name} is write-once")
        instance.__dict__[self.storage_name] = value


class P21Entity:
    identifier = P21WriteOnce()

    def __init__(self, identifier):
        self.identifier = identifier
        self.status = "new"


entity = P21Entity("E-001")
entity.status = "active"

try:
    entity.identifier = "E-002"
except AttributeError as exc:
    print(type(exc).__name__, exc)

assert entity.identifier == "E-001"
assert entity.status == "active"

AttributeError identifier is write-once


# Problem 22 — Cross-field invariant

For a date-like interval, enforce `start <= end` even though assignments happen one field at a time.

Requirements:

- either field may be assigned first during initialization;
- later updates must preserve the invariant;
- a rejected write must leave the old value unchanged.

In [22]:
class P22Interval:
    def __init__(self, start, end):
        self.start = start
        self.end = end

    def __setattr__(self, name, value):
        if name not in {"start", "end"}:
            return super().__setattr__(name, value)

        if not isinstance(value, (int, float)):
            raise TypeError(f"{name} must be numeric")

        current_start = value if name == "start" else self.__dict__.get("start")
        current_end = value if name == "end" else self.__dict__.get("end")

        if current_start is not None and current_end is not None:
            if current_start > current_end:
                raise ValueError("start must be <= end")

        super().__setattr__(name, value)


interval = P22Interval(10, 20)
interval.start = 15
interval.end = 25

try:
    interval.start = 30
except ValueError as exc:
    print(type(exc).__name__, exc)

assert interval.start == 15
assert interval.end == 25

ValueError start must be <= end


# Problem 23 — Explain descriptor precedence by experiment

Create a data descriptor and then intentionally place a conflicting key in `instance.__dict__`.

Show that reading the attribute still uses the data descriptor.

Then compare with a non-data descriptor, where an instance dictionary entry shadows the descriptor.

In [23]:
class P23Data:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return "DATA-DESCRIPTOR"

    def __set__(self, instance, value):
        instance.__dict__["data_written"] = value


class P23NonData:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return "NON-DATA-DESCRIPTOR"


class P23Demo:
    data = P23Data()
    non_data = P23NonData()


demo = P23Demo()
demo.__dict__["data"] = "INSTANCE-DICT"
demo.__dict__["non_data"] = "INSTANCE-DICT"

assert demo.data == "DATA-DESCRIPTOR"
assert demo.non_data == "INSTANCE-DICT"

demo.data = 123
assert demo.__dict__["data_written"] == 123

print("data read:", demo.data)
print("non_data read:", demo.non_data)
print("dict:", demo.__dict__)

data read: DATA-DESCRIPTOR
non_data read: INSTANCE-DICT
dict: {'data': 'INSTANCE-DICT', 'non_data': 'INSTANCE-DICT', 'data_written': 123}


# Problem 24 — Capstone: validated, audited, sealable model

Combine the techniques into one design.

Requirements:

- schema-driven validation/normalization;
- public write audit;
- optional sealing that blocks new public fields;
- existing fields remain mutable after sealing;
- private implementation attributes are not audited;
- failed writes do not appear in the audit;
- initialization uses the same public write path as later updates.

In [24]:
_P24_MISSING = object()

def p24_text(value):
    if not isinstance(value, str):
        raise TypeError("expected str")
    value = value.strip()
    if not value:
        raise ValueError("empty text is not allowed")
    return value

def p24_percentage(value):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError("expected number")
    if not 0 <= value <= 100:
        raise ValueError("percentage must be in [0, 100]")
    return float(value)


class P24Model:
    schema = {}

    def __init__(self, **values):
        object.__setattr__(self, "_audit", [])
        object.__setattr__(self, "_sealed", False)
        for name, value in values.items():
            setattr(self, name, value)

    @property
    def audit(self):
        return tuple(self._audit)

    def seal(self):
        object.__setattr__(self, "_sealed", True)

    def __setattr__(self, name, value):
        if name.startswith("_"):
            return super().__setattr__(name, value)

        schema = type(self).schema
        if name not in schema:
            raise AttributeError(f"unknown public field {name!r}")

        sealed = object.__getattribute__(self, "_sealed")
        if sealed and name not in self.__dict__:
            raise AttributeError(f"cannot add {name!r} after sealing")

        normalized = schema[name](value)  # may raise before mutation/audit
        old = self.__dict__.get(name, _P24_MISSING)

        super().__setattr__(name, normalized)
        self._audit.append((name, old, normalized))


class P24Task(P24Model):
    schema = {
        "title": p24_text,
        "progress": p24_percentage,
        "owner": p24_text,
    }


task = P24Task(title="  Deep Python ", progress=10)
task.owner = "  Ada "
task.seal()
task.progress = 55

try:
    task.progress = 101
except ValueError as exc:
    print(type(exc).__name__, exc)

try:
    task.unknown = 1
except AttributeError as exc:
    print(type(exc).__name__, exc)

assert task.title == "Deep Python"
assert task.owner == "Ada"
assert task.progress == 55.0
assert len(task.audit) == 4
assert all(event[0] != "_sealed" for event in task.audit)
print("state:", task.__dict__)
print("audit:", task.audit)

ValueError percentage must be in [0, 100]
AttributeError unknown public field 'unknown'
state: {'_audit': [('title', <object object at 0x0000026E9CB39750>, 'Deep Python'), ('progress', <object object at 0x0000026E9CB39750>, 10.0), ('owner', <object object at 0x0000026E9CB39750>, 'Ada'), ('progress', 10.0, 55.0)], '_sealed': True, 'title': 'Deep Python', 'progress': 55.0, 'owner': 'Ada'}
audit: (('title', <object object at 0x0000026E9CB39750>, 'Deep Python'), ('progress', <object object at 0x0000026E9CB39750>, 10.0), ('owner', <object object at 0x0000026E9CB39750>, 'Ada'), ('progress', 10.0, 55.0))


# Additional rapid-fire exercises

Try these without looking anything up first.

### A. Predict before running

What happens if `__setattr__` calls `setattr(self, name, value)`?

**Answer:** it re-enters `__setattr__` and normally recurses until `RecursionError`.

### B. Decide the correct delegation

When should you prefer `super().__setattr__` over `object.__setattr__`?

**Answer:** use `super()` when you want cooperative multiple inheritance and other classes in the MRO to participate. Use `object.__setattr__` when intentionally bypassing intermediate `__setattr__` implementations.

### C. Descriptor question

Does defining only `__get__` make a descriptor a data descriptor?

**Answer:** no. A data descriptor defines `__set__` and/or `__delete__` in addition to descriptor behavior.

### D. Class-vs-instance question

Does an instance's `__setattr__` intercept `MyClass.some_name = value`?

**Answer:** no. That writes to the class object and is controlled by the class's metaclass.

### E. Design question

Should all field validation live in `__setattr__`?

**Answer:** not necessarily. Descriptors are often a better fit for reusable field-level policies, while `__setattr__` is useful for cross-cutting write behavior.

# Final checklist

A production-quality attribute-write customization should be tested for:

- normal initialization;
- valid mutation;
- invalid mutation;
- unknown names;
- internal/private names;
- inheritance and `super()`;
- descriptors and properties;
- slotted classes if applicable;
- class-level writes if a metaclass is involved;
- no accidental recursion;
- rollback or audit behavior if writes have side effects.

The most important mental model is:

> **Intercept, validate/transform, then delegate exactly once.**

For class-object writes, apply the same idea at the metaclass level.